# Faza 0 — čist restart i pinovanje VIPL upstream-a


## Goal

Proveri tačan VIPL commit, licencu, mapiranje fajlova i izolaciju legacy toka pre
pokretanja bilo kakvog modela. Ovaj notebook radi na CPU-u i ne obrađuje podatke.


## Setup

Kloniraj aktivni projekat u privremeni Colab disk.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


Repo: /content/lipnet-serbian
Commit: bc7df0cd9f0499a604a33561e1fe842972cff944


## Steps

### 1. Pinuj i preuzmi tačnu upstream reviziju


In [ ]:
UPSTREAM_URL = 'https://github.com/VIPL-Audio-Visual-Speech-Understanding/LipNet-PyTorch.git'
UPSTREAM_SHA = '40209e09c49553c00c25c7d41faa3706aea3c625'
UPSTREAM = Path('/content/VIPL-LipNet-PyTorch')

remote_line = subprocess.check_output(
    ['git', 'ls-remote', UPSTREAM_URL, 'refs/heads/master'], text=True
).strip()
remote_sha = remote_line.split()[0]
print('Današnji master:', remote_sha)
if remote_sha != UPSTREAM_SHA:
    print('Napomena: master se pomerio; projekat i dalje koristi pinovani commit.')

if not (UPSTREAM / '.git').exists():
    subprocess.run(['git', 'clone', UPSTREAM_URL, str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', UPSTREAM_SHA], check=True)
checked_out = subprocess.check_output(
    ['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True
).strip()
assert checked_out == UPSTREAM_SHA
print('VIPL pin potvrđen:', checked_out)


Današnji master: 40209e09c49553c00c25c7d41faa3706aea3c625
VIPL pin potvrđen: 40209e09c49553c00c25c7d41faa3706aea3c625


### 2. Proveri inventar, licencu i evidenciju odstupanja


In [ ]:
inventory = {
    'model.py': REPO / 'lipnet/model.py',
    'dataset.py': REPO / 'lipnet/dataset.py',
    'cvtransforms.py': REPO / 'lipnet/cvtransforms.py',
    'main.py': REPO / 'lipnet/train.py',
    'demo.py': REPO / 'lipnet/demo.py',
    'options.py': REPO / 'lipnet/options.py',
}
for upstream_name, local_path in inventory.items():
    assert (UPSTREAM / upstream_name).exists(), upstream_name
    assert local_path.exists(), local_path
    print(f'{upstream_name:16s} -> {local_path.relative_to(REPO)}')

license_text = (REPO / 'lipnet/LICENSE.vipl').read_text(encoding='utf-8')
diff_text = (REPO / 'docs/upstream-diff.md').read_text(encoding='utf-8')
assert 'MIT License' in license_text and UPSTREAM_SHA in license_text
assert UPSTREAM_SHA in diff_text
print('Licenca i upstream-diff su prisutni.')


model.py         -> lipnet/model.py
dataset.py       -> lipnet/dataset.py
cvtransforms.py  -> lipnet/cvtransforms.py
main.py          -> lipnet/train.py
demo.py          -> lipnet/demo.py
options.py       -> lipnet/options.py
Licenca i upstream-diff su prisutni.


## Checks

### 3. Dokaži da aktivni kod ne zavisi od legacy manifesta/ROI modula


In [ ]:
active_files = [
    *sorted((REPO / 'lipnet').glob('*.py')),
    REPO / 'scripts/prepare_ai_speak.py',
    REPO / 'data/splits.py',
]
forbidden = ('from app', 'import app', 'manifest.csv', 'roi.csv', 'vocab.json', 'split.json')
violations = []
for path in active_files:
    text = path.read_text(encoding='utf-8')
    for token in forbidden:
        if token in text:
            violations.append((str(path.relative_to(REPO)), token))
assert not violations, violations
print('PASS: aktivni kod nema legacy import/artefakt zavisnosti.')


PASS: aktivni kod nema legacy import/artefakt zavisnosti.


In [ ]:
import json

result = {
    'phase': 0,
    'upstream_url': UPSTREAM_URL,
    'branch': 'master',
    'commit': UPSTREAM_SHA,
    'inventory': {key: str(value.relative_to(REPO)) for key, value in inventory.items()},
    'legacy_dependencies': [],
}
result_path = Path('/content/faza0_result.json')
result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(result_path.read_text())


{
  "phase": 0,
  "upstream_url": "https://github.com/VIPL-Audio-Visual-Speech-Understanding/LipNet-PyTorch.git",
  "branch": "master",
  "commit": "40209e09c49553c00c25c7d41faa3706aea3c625",
  "inventory": {
    "model.py": "lipnet/model.py",
    "dataset.py": "lipnet/dataset.py",
    "cvtransforms.py": "lipnet/cvtransforms.py",
    "main.py": "lipnet/train.py",
    "demo.py": "lipnet/demo.py",
    "options.py": "lipnet/options.py"
  },
  "legacy_dependencies": []
}



## Next Steps

Faza 0 je završena samo ako svi `assert` pozivi prođu. Zatim otvori notebook Faze 1
i uključi T4 GPU; ne prelazi na AI-SPEAK pre GRID parity provere.
